# CausalMan: causal inference data generation

This notebook generates the benchmark datasets for the CausalMan causal inference tasks.
For each (scale, seed) combination it produces:

| File | Description |
|---|---|
| `observational.csv.gz` | Training data sampled from the unmodified simulator |
| `task1_force_ltl_control/treatment.csv.gz` | Task 1 arms — intervention on `PF_M1_T1_Force_LTL` |
| `task2_force_control/treatment.csv.gz` | Task 2 arms — intervention on `PF_M1_T1_Force` |

Only observable nodes (as declared in the causal graph) are written to disk.
Each output file contains exactly `N_SAMPLES` rows.

**Edit the configuration cell below, then Run All.**

In [ ]:
# ── The only cell you need to edit ────────────────────────────────────────────

SCALE       = "micro"   # "micro" | "small" | "medium" | "large"
SEEDS       = [42]      # full benchmark: [4, 6, 42, 66, 90]
N_SAMPLES   = 10_000    # rows returned in each output CSV
OUTPUT_ROOT = "output/causalman_causal_inference"

# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import os
from pathlib import Path
import sys

import pandas as pd

# Put the repository root before this notebook directory so causalman.py
# cannot shadow the causalman package when the notebook runs in-place.
PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
                     if (path / "pyproject.toml").is_file()
                     and (path / "causalman" / "__init__.py").is_file()), None)
if PROJECT_ROOT is not None:
    project_root = str(PROJECT_ROOT)
    if project_root in sys.path:
        sys.path.remove(project_root)
    sys.path.insert(0, project_root)

from causalman import CausalMan

# Outcome variable — the binary process-result column we estimate effects on.
# Declared per scale so it can be updated if the column name differs across variants.
OUTCOME_BY_SCALE = {
    "micro":  "Sec_C2_Machine1_ProcessResult",
    "small":  "Sec_C2_Machine1_ProcessResult",
    "medium": "Sec_C2_Machine1_ProcessResult",
    "large":  "Sec_C2_Machine1_ProcessResult",
}

# Benchmark tasks — each task defines a pair of hard interventions.
# Task 1 raises the lower tolerance limit for the T1 press-fitting force.
# Task 2 raises the T1 press-fitting force itself to an abnormal level.
TASKS = [
    {
        "slug":      "task1_force_ltl",
        "variable":  "PF_M1_T1_Force_LTL",
        "control":   15000.0,
        "treatment": 18000.0,
    },
    {
        "slug":      "task2_force",
        "variable":  "PF_M1_T1_Force",
        "control":   16000.0,
        "treatment": 30000.0,
    },
]

In [ ]:
from datetime import datetime

now = datetime.now().strftime("%Y_%m_%d_%H%M%S")
OUTPUT_ROOT = f"{OUTPUT_ROOT}_{now}"

outcome = OUTCOME_BY_SCALE[SCALE]
os.makedirs(OUTPUT_ROOT, exist_ok=True)

for seed in SEEDS:
    seed_dir = os.path.join(OUTPUT_ROOT, SCALE, f"seed_{seed:03d}")
    os.makedirs(seed_dir, exist_ok=True)
    print(f"\n── {SCALE}  seed={seed} ──")

    # ── Observational data ────────────────────────────────────────────────────
    simulator = CausalMan(
        name=f"causalman_{SCALE}",
        seed=seed,
        parallelize=True,
        save_path=os.path.join(seed_dir, "observational"),
    )
    obs_dataset, _, _, _, _, _ = simulator.sample(n_samples=N_SAMPLES)

    obs_dataset.to_csv(
        os.path.join(seed_dir, "observational.csv"), index=False
    )
    print(
        f"  observational : {len(obs_dataset):,} rows, "
        f"{obs_dataset.shape[1]} observable columns saved"
    )

    # ── Interventional arms ───────────────────────────────────────────────────
    for task in TASKS:
        arms = {}
        for arm in ("control", "treatment"):
            simulator = CausalMan(
                name=f"causalman_{SCALE}",
                seed=seed,
                parallelize=True,
                save_path=os.path.join(seed_dir, str(task["slug"]), arm),
            )
            simulator.intervention_dict = {task["variable"]: task[arm]}
            interventional_dataset, _, _, _, _, _ = simulator.sample(
                n_samples=N_SAMPLES
            )

            do_str = "do(" + ",".join(
                f"{key}={value}"
                for key, value in simulator.intervention_dict.items()
            ) + ")"
            interventional_dataset.to_csv(
                os.path.join(seed_dir, f"causalman_{SCALE}_{do_str}.csv"),
                index=False,
            )
            arms[arm] = interventional_dataset

        # ATE = E[Y | do(treatment)] - E[Y | do(control)]
        ate = (
            arms["treatment"][outcome].mean()
            - arms["control"][outcome].mean()
        )
        print(f"  {task['slug']:<25} ATE = {ate:.4f}")

print(f"\nDone → {os.path.abspath(OUTPUT_ROOT)}")